In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Colab setup: Drive, code, dependencies, GPU
# ─────────────────────────────────────────────────────────────────────
# Everything this notebook produces lives under DRIVE_ROOT. Change that one
# line to move an entire experiment.
#
# Nothing is written to the container. A Colab VM can be reclaimed at any
# time and takes its local disk with it, so the dataset, the Stage 1
# sequences, checkpoints, TensorBoard events, metrics and figures all go to
# Drive. That also means a fresh session reuses the data instead of spending
# a few minutes rebuilding it.
# ═══════════════════════════════════════════════════════════════════════

DRIVE_ROOT = "/content/drive/MyDrive/hstu-shortvideo-rec"   # <-- edit me
REPO_URL   = "https://github.com/rayzhao27/hstu-shortvideo-rec.git"
REPO_DIR   = "/content/hstu-shortvideo-rec"

import os
from google.colab import drive
drive.mount('/content/drive')

RAW_DIR  = f"{DRIVE_ROOT}/raw"          # KuaiRand download
PROC_DIR = f"{DRIVE_ROOT}/processed"    # Stage 0 profile + Stage 1 sequences
PICS_DIR = f"{DRIVE_ROOT}/pictures"     # figures
RUNS_DIR = f"{DRIVE_ROOT}/runs"         # checkpoints, tb, metrics, curves
RUN_DIR  = f"{RUNS_DIR}/base"           # this run
for d in (RAW_DIR, PROC_DIR, PICS_DIR, RUNS_DIR):
    os.makedirs(d, exist_ok=True)

!git clone -q {REPO_URL} {REPO_DIR} 2>/dev/null || git -C {REPO_DIR} pull -q

# os.chdir rather than %cd: everything below is invoked as `python -m ...`, so the
# working directory has to be the repo root, and plain Python leaves no doubt about
# whether a magic expanded the variable.
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

!pip -q install -r requirements.txt
!pip -q install gin-config tensorboard
# Native fbgemm kernels are optional. utils/meta_repo.py falls back to a dense
# implementation of the three ops HSTU needs when the wheel does not match the
# runtime, so a failure on the next line is not fatal.
!pip -q install fbgemm-gpu 2>/dev/null || echo "fbgemm-gpu unavailable -> dense fallback"

!nvidia-smi
!python -m utils.check_env

Keep-awake. Prevents the **idle** disconnect only — it does not stop the session
time limit, a GPU being reclaimed, or an OOM. Those still kill the run, which is why
every checkpoint goes to Drive and why the training cell resumes from the last one.

In [ ]:
%%javascript
// Clicks the connect/reconnect control once a minute so the front-end keeps
// reporting activity to Colab.
//
// Prevents: the idle disconnect that hits a notebook nobody is interacting with.
//
// Does NOT prevent: the total session time limit, the GPU being reclaimed for a
// higher-priority job, running out of RAM or disk, or the tab being closed. Any
// of those still ends the run. Recovery is on the Drive side, not here: re-run
// the training cell and it picks up from checkpoints/last.pt.
//
// The training script also prints every --log-every steps, which keeps the cell
// producing output; that output is the second, more reliable activity signal.
function KeepAwake() {
    console.log("keep-awake ping", new Date().toISOString());
    const btn = document.querySelector("colab-toolbar-button#connect")
             || document.querySelector("colab-connect-button");
    if (btn) { btn.click(); }
}
setInterval(KeepAwake, 60000);

## Stage 0 — download and profile

Fetches KuaiRand-Pure (45MB archive, 194MB extracted), verifies its md5, prints the
schema and the profile, and writes `stats.json`, the parquet caches and `stage0_*.png`
to Drive. Both guards below skip work that is already there, so re-running this
notebook in a new session is cheap.

In [ ]:
# download is idempotent: it verifies the md5 and skips a release it already has.
!python -m data.download --raw-dir {RAW_DIR}

!test -f {PROC_DIR}/stats.json && echo "Stage 0 profile already on Drive, skipping" || \
 python -m data.explore \
   --raw-dir {RAW_DIR} \
   --processed-dir {PROC_DIR} \
   --pictures-dir {PICS_DIR} \
   --splits standard random

In [ ]:
#@title Stage 0 acceptance numbers and figures
import json
from pathlib import Path

from IPython.display import Image, display

stats = json.loads(Path(f"{PROC_DIR}/stats.json").read_text())
for split, s in stats.items():
    print(f"{split:9s} users={s['n_users']:>8,}  items={s['n_items']:>7,}  "
          f"interactions={s['n_interactions']:>10,}  mean_seq_len={s['mean_seq_len']:.1f}")

for png in sorted(Path(PICS_DIR).glob("stage0_*.png")):
    display(Image(filename=str(png)))

## Stage 1 — build the sequences

Merges the standard and random logs, drops duplicate impressions, encodes each
impression as one action, runs k-core filtering, cuts train/val/test by time, and
remaps the ids. Writes `{train,val,test}_seqs.pkl`, the encoders,
`preprocess_stats.json` and `stage1_*.png`.

Useful variations — each needs its own `--processed-dir`, or it overwrites the default
artifacts:

```bash
!python -m data.preprocess --split-strategy loo          # leave-last-one-out
!python -m data.preprocess --target-policy recommended   # bake in one stream
!python -m data.preprocess --no-random                   # biased log only
```

In [ ]:
!test -f {PROC_DIR}/train_seqs.pkl && echo "Stage 1 sequences already on Drive, skipping" || \
 python -m data.preprocess \
   --raw-dir {RAW_DIR} \
   --processed-dir {PROC_DIR} \
   --pictures-dir {PICS_DIR}

In [ ]:
#@title Stage 1 figures and a sample sequence
import json
from pathlib import Path

from IPython.display import Image, display

from data.actions import ACTION_NAMES
from data.encoders import IdEncoder
from data.sequences import load_sequences

for png in sorted(Path(PICS_DIR).glob("stage1_*.png")):
    display(Image(filename=str(png)))

item_encoder = IdEncoder.load(Path(f"{PROC_DIR}/item_encoder.pkl"))
test = load_sequences(Path(f"{PROC_DIR}/test_seqs.pkl"))

# A user with a typical amount of history, shown around their first test target.
record = sorted(test, key=lambda r: len(r["items"]))[len(test) // 2]
first_target = int(record["is_target"].argmax())
print(f"user {record['user']}: {len(record['items'])} interactions, "
      f"{record['n_targets']} test targets\n")
for i in range(max(0, first_target - 5), min(len(record["items"]), first_target + 5)):
    print(f"  {'>' if record['is_target'][i] else ' '} "
          f"item {record['items'][i]:>5} (raw {item_encoder.idx_to_raw[record['items'][i]]:>5})  "
          f"{ACTION_NAMES[record['actions'][i]]:<10} "
          f"{'random' if record['is_rand'][i] else 'recommended'}")

Audit the artifacts and print the protocol every model is scored on:

In [ ]:
!python -m data.verify --processed-dir {PROC_DIR}
!python -m data.protocol --processed-dir {PROC_DIR}

Smoke tests — run these before spending GPU hours:

In [ ]:
# models.smoke   : one forward/backward through the official HSTU encoder.
# train --smoke-test: the whole loop - train, eval, checkpoint, and a resume that
#                     is asserted to restore every tensor exactly.
!python -m models.smoke --processed-dir {PROC_DIR}
!python -m models.train --smoke-test --processed-dir {PROC_DIR} --out-dir {RUNS_DIR}/smoke

Size ladder (Stage 5 reuses these presets by name):

In [ ]:
!python -m models.config

## Stage 3 — train

**Re-running this cell after a disconnect resumes automatically** from
`checkpoints/last.pt` on Drive — it does not start over. `last.pt` is rewritten every
500 steps and at every epoch end, and `best.pt` whenever the headline metric improves.
Both are written to a temp file and renamed, so a kill mid-write cannot corrupt the
only copy.

One caveat worth knowing: **resume granularity is one epoch, not one step.** The
weights and optimizer state come back exactly as of the last save, so at most 500 steps
of *learning* are lost — but the sampler position is not stored, so restarting re-walks
the interrupted epoch from its first batch, and up to one epoch of *wall clock* can be
repeated. At batch 64 an epoch is ~400 steps.

In [ ]:
!python -m models.train \
  --size base \
  --processed-dir {PROC_DIR} \
  --out-dir {RUN_DIR} \
  --max-impressions 256 \
  --batch-size 64 \
  --epochs 20 \
  --lr 1e-3 \
  --weight-decay 0 \
  --dropout-rate 0.2 \
  --lr-schedule cosine \
  --precision bf16 \
  --grad-accum 1 \
  --eval-every 1 \
  --log-every 50 \
  --save-every-steps 500 \
  --patience 5 \
  --num-workers 2

TensorBoard (reads the event files straight off Drive):

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUN_DIR}/tb

Test metrics from the best checkpoint. Reports both protocol streams plus a
popularity baseline and the chance rate — the `random` stream is a negative control
whose targets were drawn by a uniform sampler, so chance is the expected value there,
not a model failure.

In [ ]:
!python -m models.evaluate \
  --checkpoint {RUN_DIR}/checkpoints/best.pt \
  --split      test \
  --processed-dir {PROC_DIR} \
  --batch-size 64 \
  --out        {RUN_DIR}/test_metrics.json

Checkpoint summary:

In [ ]:
import torch, json
from pathlib import Path

for name in ["last", "best"]:
    path = Path(f"{RUN_DIR}/checkpoints/{name}.pt")
    if not path.exists():
        print(f"── {name} ── missing")
        continue
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    print(f"── {name} ──")
    print("  epoch:      ", ckpt['epoch'])
    print("  step:       ", ckpt['global_step'])
    print("  best:       ", ckpt['best'])
    print("  size:       ", ckpt['config']['size'], "dim", ckpt['config']['embedding_dim'],
          "blocks", ckpt['config']['num_blocks'], "heads", ckpt['config']['num_heads'])
    print("  fingerprint:", ckpt['fingerprint'])
    print()

final = Path(f"{RUN_DIR}/final_metrics.json")
if final.exists():
    metrics = json.loads(final.read_text())
    for split in ("val", "test"):
        if split in metrics:
            rec = metrics[split]['recommended']
            pop = metrics[split]['popularity_baseline']['recommended']
            print(f"{split:5s} recommended  ndcg@10 {rec['ndcg@10']:.4f}  "
                  f"recall@10 {rec['recall@10']:.4f}   "
                  f"(popularity {pop['ndcg@10']:.4f} / {pop['recall@10']:.4f})")

Training curves (also saved to Drive as curves.png):

In [ ]:
from IPython.display import Image, display
from pathlib import Path

path = Path(f"{RUN_DIR}/curves.png")
display(Image(filename=str(path))) if path.exists() else print("no curves yet")

Scaling ladder for Stage 5 — optional and long. Each size writes its own run
directory, so each one resumes independently.

In [ ]:
# Batch sizes come from models/config.py, which sizes them for an L4 at
# max-impressions 256. On an A100 you can roughly double them.
#
# The command is assembled as a string and then run with !{cmd}: a backslash-
# continued ! invocation inside a for-loop body is not reliably parsed.
for size, batch in [("small", 128), ("base", 64), ("large", 32)]:
    print(f"\n{'=' * 70}\n{size}\n{'=' * 70}")
    cmd = (
        f"python -m models.train"
        f" --size {size}"
        f" --processed-dir {PROC_DIR}"
        f" --out-dir {RUNS_DIR}/{size}"
        f" --max-impressions 256"
        f" --batch-size {batch}"
        f" --epochs 20"
        f" --precision bf16"
        f" --save-every-steps 500"
        f" --patience 5"
        f" --num-workers 2"
    )
    !{cmd}